In [1]:
import numpy as np
import nibabel as nib
import pydicom
import matplotlib.pyplot as plt
import os
import cv2
from matplotlib.collections import LineCollection
from time import perf_counter
import pandas as pd
from scipy.ndimage import label
from tensorflow import keras
from sklearn.metrics import roc_curve, auc
from scipy.ndimage import center_of_mass
from scipy.stats import mannwhitneyu, false_discovery_control

In [2]:
def mask_into_binary(mask,index):
    #create another image matrix where the values within the greatest connected component of the target are 1 and the others are 0
    structure=np.ones((3,3,3),dtype=int)
    mask1=np.zeros(mask.shape)
    mask_i=np.array(mask==index,dtype=int)
    if np.max(mask_i)==1:
        labeled, _ = label(mask_i,structure)
        array=labeled.flatten()
        array1=array[array>0]
        counts = np.bincount(array1)
        mask1+=np.array(labeled==np.argmax(counts),dtype=int)
    return(mask1)

def find_location(mask):
    l_hip_mask=mask_into_binary(mask,77)
    r_hip_mask=mask_into_binary(mask,78)
    i0=np.max(np.array(range(mask.shape[0]))[np.max(np.max(r_hip_mask,axis=1),axis=1)==1])
    i1=np.min(np.array(range(mask.shape[0]))[np.max(np.max(l_hip_mask,axis=1),axis=1)==1])
    i=np.round((i0+i1)/2)
    j0=np.median(np.array(range(mask.shape[1]))[np.max(np.max(r_hip_mask,axis=0),axis=1)==1])
    j1=np.median(np.array(range(mask.shape[1]))[np.max(np.max(l_hip_mask,axis=0),axis=1)==1])
    j=np.round((j0+j1)/2)
    k0=np.min(np.array(range(mask.shape[2]))[np.max(r_hip_mask[i0-5,:,:],axis=0)==1])
    k1=np.min(np.array(range(mask.shape[2]))[np.max(l_hip_mask[i1+5,:,:],axis=0)==1])
    k=np.round((k0+k1)/2)
    return(int(i),int(j),int(k))

def preprocess_img(img):
    img[img<0]=0
    img[img>300]=300
    img=img/300
    return img.astype('float16')

def fiveFoldCrossValidation(patient_indexes,images,labels,k):

    x_train=[]
    y_train=[]
    x_test=[]
    y_test=[]
    test_patient_indexes=[]
    train_patient_indexes=[]
    for i in range(len(patient_indexes)):
        if patient_indexes[i] % 5==k:
            x_test.append(images[i])
            y_test.append(labels[i])
            test_patient_indexes.append(patient_indexes[i])
        else:
            x_train.append(images[i])
            y_train.append(labels[i])
            train_patient_indexes.append(patient_indexes[i])
    x_train=np.array(x_train).astype('float16')
    y_train=np.array(y_train) 
    x_test=np.array(x_test).astype('float16')
    y_test=np.array(y_test)
    return(x_train,y_train,x_test,y_test,train_patient_indexes,test_patient_indexes)

def train_classifier(x_train,y_train,numberOfEpochs,name):
    
    img_height=x_train[0].shape[0]
    model = keras.Sequential([keras.layers.Conv2D(16, 3, activation='relu', input_shape=(img_height,img_height,1)),
                        keras.layers.Conv2D(16, 3, activation='relu'),
                        keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
                        keras.layers.Conv2D(32, 3, activation='relu'),
                        keras.layers.Conv2D(32, 3, activation='relu'),
                        keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
                        keras.layers.Conv2D(64, 3, activation='relu'),
                        keras.layers.Conv2D(64, 3, activation='relu'),
                        keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
                        keras.layers.Conv2D(128, 3, activation='relu'),
                        keras.layers.Conv2D(128, 3, activation='relu'),
                        keras.layers.MaxPooling2D(strides=(2, 2)),
                        keras.layers.Flatten(),
                        keras.layers.Dense(128, activation='relu'),
                        keras.layers.Dense(64, activation='relu'),
                        keras.layers.Dense(32, activation='relu'),
                        keras.layers.Dense(1, activation='sigmoid')
    ]
    )
    model.compile(
            optimizer=keras.optimizers.Adam(1e-3),
            loss=keras.losses.BinaryCrossentropy(),
            metrics=[keras.metrics.BinaryAccuracy()]
    )
    history=model.fit(x=x_train,y=y_train,epochs=numberOfEpochs,shuffle=True)
    model.save('{}.keras'.format(name),include_optimizer=True)
    plt.plot(range(len(history.history['loss'])),history.history['loss'],color='blue')

def compute_patientwise_results(predictions,y_test,patientIndexes_test):

    predictions_1=[]
    y_test_1=[]
    for i in range(len(np.unique(patientIndexes_test))):
        predictions_1.append(np.mean(predictions[patientIndexes_test==np.unique(patientIndexes_test)[i]]))
        y_test_1.append(np.max(y_test[patientIndexes_test==np.unique(patientIndexes_test)[i]])) 
    predictions_1=np.array(predictions_1)
    y_test_1=np.array(y_test_1)   
    return predictions_1, y_test_1

def evaluate_predictions(predictions,y_test,train_predictions,y_train):

    fpr, tpr, thresholds = roc_curve(y_train,train_predictions,drop_intermediate=False)
    J_stats = tpr - fpr
    youden = thresholds[np.argmax(J_stats)]
    TP = np.sum(y_test[predictions>=youden])
    FN = np.sum(y_test[predictions<youden])
    FP = np.sum((1-y_test)[predictions>=youden])
    TN = np.sum((1-y_test)[predictions<youden])
    acc = (TP+TN)/(TP+TN+FN+FP) 
    sen = TP/(TP+FN)
    spe = TN/(TN+FP)
    pre = TP/(TP+FP)
    fpr, tpr, thresholds = roc_curve(y_test,predictions,drop_intermediate=False)
    auc1=auc(fpr, tpr)
    return(acc,sen,spe,pre,auc1)

def evaluate_predictions_1(predictions,y_test,test_patient_indexes,train_predictions,y_train,train_patient_indexes):
    acc,sen,spe,pre,auc1=evaluate_predictions(predictions,y_test,train_predictions,y_train)
    predictions_1, y_test_1=compute_patientwise_results(predictions,y_test,test_patient_indexes)
    train_predictions_1, y_train_1=compute_patientwise_results(train_predictions,y_train,train_patient_indexes)
    acc_1,sen_1,spe_1,pre_1,auc1_1=evaluate_predictions(predictions_1,y_test_1,train_predictions_1,y_train_1)
    return(acc,sen,spe,pre,auc1,acc_1,sen_1,spe_1,pre_1,auc1_1) 

#functions for visualization
def get_all_edges(bool_img):
    """
    Get a list of all edges (where the value changes from True to False) in the 2D boolean image.
    The returned array edges has he dimension (n, 2, 2).
    Edge i connects the pixels edges[i, 0, :] and edges[i, 1, :].
    Note that the indices of a pixel also denote the coordinates of its lower left corner.
    """
    edges = []
    ii, jj = np.nonzero(bool_img)
    for i, j in zip(ii, jj):
        # North
        if j == bool_img.shape[1]-1 or not bool_img[i, j+1]:
            edges.append(np.array([[i, j+1],
                                   [i+1, j+1]]))
        # East
        if i == bool_img.shape[0]-1 or not bool_img[i+1, j]:
            edges.append(np.array([[i+1, j],
                                   [i+1, j+1]]))
        # South
        if j == 0 or not bool_img[i, j-1]:
            edges.append(np.array([[i, j],
                                   [i+1, j]]))
        # West
        if i == 0 or not bool_img[i-1, j]:
            edges.append(np.array([[i, j],
                                   [i, j+1]]))

    if not edges:
        return np.zeros((0, 2, 2))
    else:
        return np.array(edges)


def close_loop_edges(edges):
    """
    Combine the edges defined by 'get_all_edges' to closed loops around objects.
    If there are multiple disconnected objects a list of closed loops is returned.
    Note that it's expected that all the edges are part of exactly one loop (but not necessarily the same one).
    """

    loop_list = []
    while edges.size != 0:

        loop = [edges[0, 0], edges[0, 1]]  # Start with first edge
        edges = np.delete(edges, 0, axis=0)

        while edges.size != 0:
            # Get next edge (=edge with common node)
            ij = np.nonzero((edges == loop[-1]).all(axis=2))
            if ij[0].size > 0:
                i = ij[0][0]
                j = ij[1][0]
            else:
                loop.append(loop[0])
                # Uncomment to to make the start of the loop invisible when plotting
                # loop.append(loop[1])
                break

            loop.append(edges[i, (j + 1) % 2, :])
            edges = np.delete(edges, i, axis=0)

        loop_list.append(np.array(loop))

    return loop_list


def plot_outlines(bool_img, ax=None, **kwargs):
    if ax is None:
        ax = plt.gca()
    edges = get_all_edges(bool_img=bool_img)
    edges = edges - 0.5  # convert indices to coordinates; TODO adjust according to image extent
    outlines = close_loop_edges(edges=edges)
    cl = LineCollection(outlines, **kwargs)
    ax.add_collection(cl)

In [ ]:
excel=pd.read_excel('C:/Users/oonar/Documents/yct/prostate_status.xlsx')
indexes_0=[]
indexes_1=[]
for i in range(excel.shape[0]):
    if excel['prostate'][i]=='intact':
        indexes_0.append(excel['bcr index'][i])
    if excel['prostate'][i]=='removed':
        indexes_1.append(excel['bcr index'][i])
print(len(indexes_0))
print(len(indexes_1))

In [ ]:
info=np.zeros((len(indexes_0),14))
for i in range(len(indexes_0)):
    mask_path='D:/prostateCancer/bcr_niftis/ts/ts_{}.nii'.format(indexes_0[i])
    mask=nib.load(mask_path)
    affine=mask.affine
    mask=np.round(mask.get_fdata())
    info[i,0]=indexes_0[i]
    info[i,1]=mask.shape[0]
    info[i,2]=mask.shape[1]
    info[i,3]=mask.shape[2]
    info[i,4]=abs(affine[0,0])
    info[i,5]=abs(affine[1,1])
    info[i,6]=abs(affine[2,2])
    if np.max(np.array(mask==22,dtype='int'))==1:
        mask1=mask_into_binary(mask,22)
        com=center_of_mass(mask1)
        info[i,7]=com[0]
        info[i,8]=com[1]
        info[i,9]=com[2]
        info[i,13]=np.sum(mask1)
    if np.sum(np.array(mask==77,dtype='int'))>1000 and np.sum(np.array(mask==78,dtype='int'))>1000:
        i1,j1,k1=find_location(mask)
        info[i,10]=i1
        info[i,11]=j1
        info[i,12]=k1
    print(i)
np.savetxt('info_indexes_0.csv',info)

info=np.zeros((len(indexes_1),14))
for i in range(len(indexes_1)):
    mask_path='D:/prostateCancer/bcr_niftis/ts/ts_{}.nii'.format(indexes_1[i])
    mask=nib.load(mask_path)
    affine=mask.affine
    mask=np.round(mask.get_fdata())
    info[i,0]=indexes_1[i]
    info[i,1]=mask.shape[0]
    info[i,2]=mask.shape[1]
    info[i,3]=mask.shape[2]
    info[i,4]=abs(affine[0,0])
    info[i,5]=abs(affine[1,1])
    info[i,6]=abs(affine[2,2])
    if np.max(np.array(mask==22,dtype='int'))==1:
        mask1=mask_into_binary(mask,22)
        com=center_of_mass(mask1)
        info[i,7]=com[0]
        info[i,8]=com[1]
        info[i,9]=com[2]
        info[i,13]=np.sum(mask1)
    if np.sum(np.array(mask==77,dtype='int'))>1000 and np.sum(np.array(mask==78,dtype='int'))>1000:
        i1,j1,k1=find_location(mask)
        info[i,10]=i1
        info[i,11]=j1
        info[i,12]=k1
    print(i)
np.savetxt('info_indexes_1.csv',info)

folders=os.listdir('D:/prostateCancer/carimas_files')
info=np.zeros((len(folders),14))
for i in range(len(folders)):
    mask_path='D:/prostateCancer/ts_prostage/ts_{}.nii'.format(folders[i])
    mask=nib.load(mask_path)
    affine=mask.affine
    mask=np.round(mask.get_fdata())
    info[i,0]=int(folders[i])
    info[i,1]=mask.shape[0]
    info[i,2]=mask.shape[1]
    info[i,3]=mask.shape[2]
    info[i,4]=abs(affine[0,0])
    info[i,5]=abs(affine[1,1])
    info[i,6]=abs(affine[2,2])
    if np.max(np.array(mask==22,dtype='int'))==1:
        mask1=mask_into_binary(mask,22)
        com=center_of_mass(mask1)
        info[i,7]=com[0]
        info[i,8]=com[1]
        info[i,9]=com[2]
        info[i,13]=np.sum(mask1)
    if np.sum(np.array(mask==77,dtype='int'))>1000 and np.sum(np.array(mask==78,dtype='int'))>1000:
        i1,j1,k1=find_location(mask)
        info[i,10]=i1
        info[i,11]=j1
        info[i,12]=k1
    print(i)
np.savetxt('info_3rd_dataset.csv',info)

In [ ]:
info=np.loadtxt('info_indexes_1.csv')
print(np.max(info[:,3][info[:,4]>0.9]))

#512*512*263 - 512*512*671
#512*512*263 - 512*512*629
#512*512*341 - 512*512*395

#0.941*0.941*1.5 - 1.523*1.523*3.270
#0.977*0.977*1.5 - 1.523*1.523*3.270
#0.973*0.973*2.790 - 1.367*1.367*2.790

In [ ]:
images=[]
labels=[]
patient_indexes=[]
structure=np.ones((3,3,3),dtype=int)
con1=[]
con2=[]
con3=[]

for i in range(len(indexes_0)):
    img_path='D:/prostateCancer/bcr_niftis/ct/ct_{}.nii'.format(indexes_0[i])
    mask_path='D:/prostateCancer/bcr_niftis/ts/ts_{}.nii'.format(indexes_0[i])
    img=nib.load(img_path).get_fdata()
    mask=np.round(nib.load(mask_path).get_fdata())
    if np.max(np.array(mask==22,dtype='int'))==1:
        mask1=mask_into_binary(mask,22)
        com=center_of_mass(mask1)
        i1=int(np.round(com[0]))
        j1=int(np.round(com[1]))
        k1=int(np.round(com[2]))
    else:
        i1,j1,k1=find_location(mask)
    for j in range(10):
        images.append(preprocess_img(img[i1-64:i1+64,j1-64:j1+64,k1-5+j]))
        labels.append(0)
        patient_indexes.append(indexes_0[i]) 
    mask_i=np.array(mask==22,dtype=int)
    if np.max(mask_i)==1:
        labeled, num = label(mask_i,structure)
        con2.append(num)
    else:
        con2.append(0)
    print(i)

for i in range(len(indexes_1)):
    if indexes_1[i]!=47:
        img_path='D:/prostateCancer/bcr_niftis/ct/ct_{}.nii'.format(indexes_1[i])
        mask_path='D:/prostateCancer/bcr_niftis/ts/ts_{}.nii'.format(indexes_1[i])
        img=nib.load(img_path).get_fdata()
        mask=np.round(nib.load(mask_path).get_fdata())
        if np.max(np.array(mask==22,dtype='int'))==1:
            mask1=mask_into_binary(mask,22)
            com=center_of_mass(mask1)
            i1=int(np.round(com[0]))
            j1=int(np.round(com[1]))
            k1=int(np.round(com[2]))
        else:
            i1,j1,k1=find_location(mask)
        for j in range(10):
            images.append(preprocess_img(img[i1-64:i1+64,j1-64:j1+64,k1-5+j]))
            labels.append(1)
            patient_indexes.append(indexes_1[i]) 
        mask_i=np.array(mask==22,dtype=int)
        if np.max(mask_i)==1:
            labeled, num = label(mask_i,structure)
            con1.append(num)
        else:
            con1.append(0)
        print(i)

folders=os.listdir('D:/prostateCancer/carimas_files')
for i in range(len(folders)):
    img_path='D:/prostateCancer/carimas_files/{}/ct.nifti.img'.format(folders[i])
    mask_path='D:/prostateCancer/ts_prostage/ts_{}.nii'.format(folders[i])
    img=nib.load(img_path).get_fdata()
    mask=np.round(nib.load(mask_path).get_fdata())
    if np.max(np.array(mask==22,dtype='int'))==1:
        mask1=mask_into_binary(mask,22)
        com=center_of_mass(mask1)
        i1=int(np.round(com[0]))
        j1=int(np.round(com[1]))
        k1=int(np.round(com[2]))
    else:
        i1,j1,k1=find_location(mask)
    for j in range(10):
        images.append(preprocess_img(img[i1-64:i1+64,j1-64:j1+64,k1-5+j]))
        labels.append(0)
        patient_indexes.append(1000+int(folders[i]))
    mask_i=np.array(mask==22,dtype=int)
    if np.max(mask_i)==1:
        labeled, num = label(mask_i,structure)
        con3.append(num)
    else:
        con3.append(0)
    print(i)

In [ ]:
np.savetxt('con1.csv',np.array(con1))
np.savetxt('con2.csv',np.array(con2))
np.savetxt('con3.csv',np.array(con3))

In [ ]:
results=np.zeros((5,10))
np.savetxt('patient_indexes.csv',patient_indexes)
np.savetxt('labels.csv',labels)
numberOfEpochs=10
for k in range(5):
    x_train,y_train,x_test,y_test,train_patient_indexes,test_patient_indexes=fiveFoldCrossValidation(patient_indexes,images,labels,k)
    name='classifier_trans_{}'.format(k)
    train_classifier(x_train,y_train,numberOfEpochs,name)
    classifier=keras.models.load_model('{}.keras'.format(name))
    classifier.compile(optimizer='adam',loss='BinaryCrossentropy')
    test_predictions=classifier.predict(x_test)[:,0]
    train_predictions=classifier.predict(x_train)[:,0]
    np.savetxt('test_predictions_new_{}.csv'.format(k),test_predictions)
    np.savetxt('train_predictions_new_{}.csv'.format(k),train_predictions)
    results[k,:]=evaluate_predictions_1(test_predictions,y_test,test_patient_indexes,train_predictions,y_train,train_patient_indexes)
    predictions_trans, y_test_1=compute_patientwise_results(test_predictions,y_test,test_patient_indexes)
    train_predictions_trans, y_train_1=compute_patientwise_results(train_predictions,y_train,train_patient_indexes)
np.savetxt('results_new.csv',results)

In [ ]:
results=np.loadtxt('results_new.csv')
print(results)
for i in range(10):
    print(np.mean(results[:,i]),np.std(results[:,i]))

In [ ]:
patient_indexes=np.loadtxt('patient_indexes.csv')
labels=np.loadtxt('labels.csv')
for k in range(5):
    test_predictions=np.loadtxt('test_predictions_new_{}.csv'.format(k))
    images=labels
    x_train,y_train,x_test,y_test,train_patient_indexes,test_patient_indexes=fiveFoldCrossValidation(patient_indexes,images,labels,k)
    predictions_trans, y_test_1=compute_patientwise_results(test_predictions,y_test,test_patient_indexes)
    print(evaluate_predictions(predictions_trans, y_test_1,predictions_trans, y_test_1))
    np.savetxt('y_{}_for_r.csv'.format(k),y_test_1)
    np.savetxt('preds_{}_for_r.csv'.format(k),predictions_trans)

In [ ]:
i=0
if indexes_1[i]!=47:
    img_path='D:/prostateCancer/bcr_niftis/ct/ct_{}.nii'.format(indexes_1[i])
    mask_path='D:/prostateCancer/bcr_niftis/ts/ts_{}.nii'.format(indexes_1[i])
    img=nib.load(img_path).get_fdata()
    mask=np.round(nib.load(mask_path).get_fdata())
    mask=np.array(mask==22,dtype='int')
    img[img<0]=0
    img[img>300]=300

In [ ]:
ind=np.argmax(np.sum(np.sum(mask,axis=0),axis=0))
plt.imshow(np.transpose(img[:,:,ind]),cmap='gray')
plt.colorbar(label='Hounsfield unit')
plot_outlines(np.transpose(mask[:,:,ind]).T,color='lightblue')
plt.axis('off')
fig=plt.gcf()
fig.savefig('fig3b.png',bbox_inches='tight')
plt.show()

In [ ]:
folders=os.listdir('D:/prostateCancer/carimas_files')
i=0
img_path='D:/prostateCancer/carimas_files/{}/ct.nifti.img'.format(folders[i])
mask_path='D:/prostateCancer/ts_prostage/ts_{}.nii'.format(folders[i])
img=nib.load(img_path).get_fdata()
mask=np.round(nib.load(mask_path).get_fdata())
mask=np.array(mask==22,dtype='int')
img[img<0]=0
img[img>300]=300

In [ ]:
ind=np.argmax(np.sum(np.sum(mask,axis=0),axis=0))
plt.imshow(np.transpose(img[:,:,ind]),cmap='gray')
plt.colorbar(label='Hounsfield unit')
plot_outlines(np.transpose(mask[:,:,ind]).T,color='lightblue')
plt.axis('off')
fig=plt.gcf()
fig.savefig('fig3a.png',bbox_inches='tight')
plt.show()

In [ ]:
con3=np.loadtxt('con3.csv')
print(len(con3))
print(con3)

In [ ]:
info=np.loadtxt('info_3rd_dataset.csv')
print(info[3,])

In [ ]:
folders=os.listdir('D:/prostateCancer/carimas_files')
i=3
img_path='D:/prostateCancer/carimas_files/{}/ct.nifti.img'.format(folders[i])
mask_path='D:/prostateCancer/ts_prostage/ts_{}.nii'.format(folders[i])
img=nib.load(img_path).get_fdata()
mask=np.round(nib.load(mask_path).get_fdata())
mask=np.array(mask==22,dtype='int')
img[img<0]=0
img[img>300]=300

In [ ]:
print(np.argmax(np.sum(np.sum(mask,axis=2),axis=0)))

In [ ]:
img1=cv2.resize(np.mean(img,axis=1),(int(np.round(395*2.78996873/1.138)),512))
mask1=cv2.resize(np.array(np.max(mask,axis=1),dtype='uint8'),(int(np.round(395*2.78996873/1.138)),512))
#plt.imshow(np.max(mask,axis=1),cmap='gray')
plt.imshow(np.flip(np.transpose(img1),0),cmap='gray')
plot_outlines(np.flip(np.transpose(mask1),0).T,color='white')
plt.axis('off')
fig=plt.gcf()
fig.savefig('fig3c.png',bbox_inches='tight')
plt.show()

In [ ]:
#img=img[100:412,100:412,:]
kuv=np.ones((int(img.shape[0]/2)+int(395*2.78996873/1.138),int(1.5*img.shape[0])))
for k in range(int(img.shape[0]/2)):
    img_i=np.transpose(img[:,2*k,:])
    img_i=cv2.resize(img_i,(312,int(395*2.78996873/1.138)))
    kuv[int(img.shape[0]/2)-k:int(img.shape[0]/2)-k+img_i.shape[0],k:img_i.shape[1]+k]+=img_i
kuv=kuv/np.max(kuv)
for k in range(int(img.shape[0]/2)):
    kuv[0:(int(img.shape[0]/2)-k),0:k]=1
    kuv[img_i.shape[0]+k:img_i.shape[0]+int(img.shape[0]/2),int(1.5*img.shape[0])-k:int(1.5*img.shape[0])]=1
    kuv[int(img.shape[0]/2)-k:int(img.shape[0]/2)-k+3,img.shape[0]+k:img.shape[0]+k+4]=1
kuv[int(img.shape[0]/2):int(img.shape[0]/2)+img_i.shape[0],img.shape[0]:img.shape[0]+3]=1
kuv[int(img.shape[0]/2):int(img.shape[0]/2)+3,0:img.shape[0]]=1
plt.imshow(kuv,cmap='gray')
plt.axis('off')
fig=plt.gcf()
fig.savefig('fig1.png',bbox_inches='tight')

In [ ]:
img=mask[100:412,100:412,:]
kuv=np.ones((int(img.shape[0]/2)+int(395*2.78996873/1.138),int(1.5*img.shape[0])))
for k in range(int(img.shape[0]/2)):
    img_i=np.transpose(img[:,2*k,:])
    img_i=cv2.resize(img_i,(312,int(395*2.78996873/1.138)))
    kuv[int(img.shape[0]/2)-k:int(img.shape[0]/2)-k+img_i.shape[0],k:img_i.shape[1]+k]+=img_i
kuv=kuv/np.max(kuv)
for k in range(int(img.shape[0]/2)):
    kuv[0:(int(img.shape[0]/2)-k),0:k]=1
    kuv[img_i.shape[0]+k:img_i.shape[0]+int(img.shape[0]/2),int(1.5*img.shape[0])-k:int(1.5*img.shape[0])]=1
    kuv[int(img.shape[0]/2)-k:int(img.shape[0]/2)-k+3,img.shape[0]+k:img.shape[0]+k+4]=1
kuv[int(img.shape[0]/2):int(img.shape[0]/2)+img_i.shape[0],img.shape[0]:img.shape[0]+3]=1
kuv[int(img.shape[0]/2):int(img.shape[0]/2)+3,0:img.shape[0]]=1
plt.imshow(kuv,cmap='gray')
plt.axis('off')
fig=plt.gcf()
fig.savefig('fig2.png',bbox_inches='tight')

In [ ]:
i=0
images=[]
img_path='D:/prostateCancer/bcr_niftis/ct/ct_{}.nii'.format(indexes_1[i])
mask_path='D:/prostateCancer/bcr_niftis/ts/ts_{}.nii'.format(indexes_1[i])
img=nib.load(img_path).get_fdata()
mask=np.round(nib.load(mask_path).get_fdata())
if np.max(np.array(mask==22,dtype='int'))==1:
    mask1=mask_into_binary(mask,22)
    com=center_of_mass(mask1)
    i1=int(np.round(com[0]))
    j1=int(np.round(com[1]))
    k1=int(np.round(com[2]))
for j in range(10):
    images.append(img[i1-64:i1+64,j1-64:j1+64,k1-5+j])

In [ ]:
kuv=np.ones((650,450))
for k in range(10):
    img_i=cv2.resize(np.transpose(images[k]),(300,300))
    img_i=img_i/np.max(img_i)
    img_i[0:300,290:300]=1
    img_i[290:300,0:300]=1
    for i in range(300):
        for j in range(300):
            kuv[int(i/2)+500-50*k,j+150-int(i/2)]=img_i[i,j]
plt.imshow(kuv,cmap='gray')
plt.axis('off')
fig=plt.gcf()
fig.savefig('fig3.png',bbox_inches='tight')

In [ ]:
kuv=np.ones((450,150))
k=0
img_i=cv2.resize(np.transpose(images[9]),(300,300))
img_i=img_i/np.max(img_i)
img_i[0:300,290:300]=1
img_i[290:300,0:300]=1
for i in range(300):
    for j in range(300):
        kuv[i+150-int(j/2),int(j/2)]=img_i[i,j]
plt.imshow(kuv,cmap='gray')
plt.axis('off')
fig=plt.gcf()
fig.savefig('fig4.png',bbox_inches='tight')

In [ ]:
y=np.loadtxt('C:/Users/oonar/Documents/yct/y_0_for_r.csv')
x=np.loadtxt('C:/Users/oonar/Documents/yct/preds_0_for_r.csv')
z=np.loadtxt('C:/Users/oonar/Documents/yct/z_0_for_r.csv')
y1=np.loadtxt('C:/Users/oonar/Documents/yct/y1_0_for_r.csv')
fpr1, tpr1, thresholds1 = roc_curve(y,x,drop_intermediate=False)
fpr0, tpr0, thresholds0 = roc_curve(y1,z,drop_intermediate=False)
plt.plot(fpr0, tpr0,color='lightblue')
plt.plot(fpr1, tpr1,color='blue')
plt.ylabel('True positive rate')
plt.xlabel('False positive rate')
fig=plt.gcf()
fig.savefig('fig5.png',bbox_inches='tight')

In [ ]:
patient_indexes=np.loadtxt('patient_indexes.csv')
labels=np.loadtxt('labels.csv')
k=3
test_predictions=np.loadtxt('test_predictions_new_{}.csv'.format(k))
train_predictions=np.loadtxt('train_predictions_new_{}.csv'.format(k))
images=labels
x_train,y_train,x_test,y_test,train_patient_indexes,test_patient_indexes=fiveFoldCrossValidation(patient_indexes,images,labels,k)
predictions_trans, y_test_1=compute_patientwise_results(test_predictions,y_test,test_patient_indexes)
predictions_trans_train, y_train_1=compute_patientwise_results(train_predictions,y_train,train_patient_indexes)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_train_1,predictions_trans_train,drop_intermediate=False)
J_stats = tpr - fpr
youden = thresholds[np.argmax(J_stats)]
TP = np.sum(y_test_1[predictions_trans>=youden])
FN = np.sum(y_test_1[predictions_trans<youden])
FP = np.sum((1-y_test_1)[predictions_trans>=youden])
TN = np.sum((1-y_test_1)[predictions_trans<youden])
print(TP,FN,FP,TN)

In [ ]:
print(np.unique(test_patient_indexes)[np.logical_and(y_test_1==0,predictions_trans>=youden)])

In [ ]:
for i in [211,271,356,376,1046,48,363,383]:
    if i<1000:
        img_path='D:/prostateCancer/bcr_niftis/ct/ct_{}.nii'.format(i)
        mask_path='D:/prostateCancer/bcr_niftis/ts/ts_{}.nii'.format(i)
    else:
        img_path='D:/prostateCancer/carimas_files/{}/ct.nifti.img'.format(i-1000)
        mask_path='D:/prostateCancer/ts_prostage/ts_{}.nii'.format(i-1000)
    img=nib.load(img_path).get_fdata()
    mask=np.round(nib.load(mask_path).get_fdata())
    if np.max(np.array(mask==22,dtype='int'))==1:
        mask1=mask_into_binary(mask,22)
        com=center_of_mass(mask1)
        i1=int(np.round(com[0]))
        j1=int(np.round(com[1]))
        k1=int(np.round(com[2]))
    else:
        i1,j1,k1=find_location(mask)
    img_i=preprocess_img(img[i1-64:i1+64,j1-64:j1+64,k1])
    plt.imshow(np.transpose(img_i),cmap='gray')
    plt.axis('off')
    fig=plt.gcf()
    fig.savefig('fp_{}.png'.format(i),bbox_inches='tight')
    plt.show()

In [ ]:
ages=[]
scanners=[]
times=[]
for i in range(len(indexes_1)):#
    path_to_images='D:/prostateCancer/BCR/{}'.format(indexes_1[i])
    files=os.listdir(path_to_images)
    for j in range(len(files)):
        if os.path.isdir('{}/{}'.format(path_to_images,files[j])):
            path_to_dicom='{}/{}'.format(path_to_images,files[j])
            path_to_1st='{}/{}'.format(path_to_dicom,os.listdir(path_to_dicom)[0])
            ds=pydicom.dcmread(path_to_1st,force=True)
            if 'Modality' in ds:
                if ds.Modality=='CT':
                    ages.append(int(ds.PatientAge[1:3]))
                    scanners.append(ds.ManufacturerModelName)
                    times.append(int(ds.StudyDate))

In [ ]:
print(np.mean(ages),np.std(ages))

In [ ]:
print(np.max(times))

In [ ]:
print(np.unique(scanners))

In [ ]:
60*0.2306392733337512

In [ ]:
results=np.zeros((5,10))
np.savetxt('patient_indexes.csv',patient_indexes)
np.savetxt('labels.csv',labels)
numberOfEpochs=10
for k in range(1):
    x_train,y_train,x_test,y_test,train_patient_indexes,test_patient_indexes=fiveFoldCrossValidation(patient_indexes,images,labels,k)
    name='classifier_trans_{}_time'.format(k)
    startTime=perf_counter()
    train_classifier(x_train,y_train,numberOfEpochs,name)
    endTime=perf_counter()
    processingTime=endTime-startTime
    print(processingTime/60)
    classifier=keras.models.load_model('{}.keras'.format(name))
    classifier.compile(optimizer='adam',loss='BinaryCrossentropy')
    test_predictions=classifier.predict(x_test[0:10])[:,0]